# 01 - Data Cleaning

Load the raw CSV, inspect quality, handle outliers, and save the cleaned file.

## 1. Load raw data

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import config
from src.data_loader import load_raw

df = load_raw()
print(f"Raw data shape: {df.shape}")
df.head()

Raw data shape: (10000, 8)


,user_id,group,converted,revenue,session_time_min,pages_viewed,device,country
0,1001,experiment,0,0.0,5.15,2,desktop,US
1,1002,control,0,0.0,8.34,2,mobile,US
2,1003,control,0,0.0,6.65,4,mobile,UK
3,1004,control,0,0.0,7.14,2,desktop,CA
4,1005,control,0,0.0,18.00,1,desktop,US


## 2. Inspect data

In [3]:
print("Missing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)

Missing values:
 user_id             0
group               0
converted           0
revenue             0
session_time_min    0
pages_viewed        0
device              0
country             0
dtype: int64

Data types:
 user_id               int64
group                object
converted             int64
revenue             float64
session_time_min    float64
pages_viewed          int64
device               object
country              object
dtype: object


## 3. Group split check

In [4]:
print(df['group'].value_counts())
print("\nDevice distribution:")
print(df['device'].value_counts())
print("\nCountry distribution:")
print(df['country'].value_counts())

group
experiment    5200
control       4800
Name: count, dtype: int64

Device distribution:
device
desktop    5200
mobile     3825
tablet      975
Name: count, dtype: int64

Country distribution:
country
US    5527
UK    1805
CA    1205
AU     794
IN     669
Name: count, dtype: int64


## 4. Outlier handling — revenue cap at 99th percentile

In [5]:
q99 = df['revenue'].quantile(0.99)
print(f"99th percentile of revenue: ${q99:.2f}")
print(f"Max before cap: ${df['revenue'].max():.2f}")

df_clean = df.drop_duplicates(subset='user_id').copy()
df_clean['revenue'] = df_clean['revenue'].clip(upper=q99)

print(f"Max after cap : ${df_clean['revenue'].max():.2f}")
print(f"Rows kept     : {len(df_clean):,}")

99th percentile of revenue: $121.73
Max before cap: $495.58
Max after cap : $121.73
Rows kept     : 10,000


## 5. Save cleaned data

In [6]:
output_path = PROJECT_ROOT / config.DATA_CLEANED
df_clean.to_csv(output_path, index=False)
print("Saved:", output_path)
df_clean.describe().round(3)

Saved: C:\60-days-data-science\day40-ab-testing\data\ab_test_cleaned.csv


,user_id,converted,revenue,session_time_min,pages_viewed
count,10000.000,10000.000,10000.000,10000.000,10000.000
mean,6000.500,0.129,7.206,9.772,4.363
std,2886.896,0.335,22.037,4.757,1.865
min,1001.000,0.000,0.000,0.660,1.000
25%,3500.750,0.000,0.000,6.310,3.000
50%,6000.500,0.000,0.000,8.940,4.000
75%,8500.250,0.000,0.000,12.332,5.000
max,11000.000,1.000,121.733,35.260,13.000
